# 05 — Classification and Baselines

This notebook evaluates the distance-based features generated in Notebook 04.

Models:
1. SVM using the mined-pattern similarity features.
2. ANN baseline using the same feature matrix.
3. A simple threshold-based baseline using maximum pattern similarity.

The split is performed at the patient-window level using stratification and a fixed random seed.

This notebook focuses on model training and validation. Detailed metrics, plots, and ablation analysis are handled in Notebook 06.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "outputs").exists() and (PROJECT_ROOT.parent / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

FEATURE_DIR = PROJECT_ROOT / "outputs" / "features"
MODEL_DIR = PROJECT_ROOT / "outputs" / "models"
EVAL_DIR = PROJECT_ROOT / "outputs" / "evaluation"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_FILE = FEATURE_DIR / "sequence_distance_features.csv"

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Project root:", PROJECT_ROOT)
print("Feature file:", FEATURE_FILE)


## 1. Load the distance features

Notebook 04 saved the complete feature matrix. The patient identifier is retained for traceability but is not used as a predictive feature.

The default model features are the similarity columns plus the aggregate similarity features.


In [ ]:
if not FEATURE_FILE.exists():
    raise FileNotFoundError(
        f"Run Notebook 04 first. Missing: {FEATURE_FILE}"
    )

features_df = pd.read_csv(FEATURE_FILE)

print("Dataset shape:", features_df.shape)
display(features_df.head())

similarity_columns = [
    c for c in features_df.columns
    if c.endswith("_similarity")
]

if not similarity_columns:
    raise ValueError("No similarity features found.")

X = features_df[similarity_columns].copy()
y = features_df["label"].astype(int).copy()

print("Feature columns:")
print(similarity_columns)

print("\nClass distribution:")
print(y.value_counts())


## 2. Train/test split

A stratified split preserves the positive/negative class proportions.

For a future rigorous experiment, the project should ideally use patient-level splitting. The current sequence files contain one window per patient record, so this split is also patient-window independent at the current dataset construction level.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True).sort_index())


## 3. Evaluation helper

Because sepsis prediction is a binary classification problem and the classes may not be perfectly balanced, we report:

- Precision
- Recall
- F1
- AUROC
- AUPRC

Accuracy is also reported for completeness.


In [ ]:
def evaluate_model(name, y_true, y_pred, y_score):
    result = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auroc": roc_auc_score(y_true, y_score),
        "auprc": average_precision_score(y_true, y_score)
    }

    print(f"\n===== {name} =====")
    print(f"Accuracy : {result['accuracy']:.4f}")
    print(f"Precision: {result['precision']:.4f}")
    print(f"Recall   : {result['recall']:.4f}")
    print(f"F1       : {result['f1']:.4f}")
    print(f"AUROC    : {result['auroc']:.4f}")
    print(f"AUPRC    : {result['auprc']:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["Non-sepsis", "Pre-sepsis"],
            zero_division=0
        )
    )

    return result


## 4. SVM classifier

The proposal specifies an SVM using the distance-derived feature representation.

A standard RBF SVM is used here first. Scaling is included in the pipeline so the model receives normalized features.

`probability=True` provides probability estimates for AUROC/AUPRC evaluation.


In [ ]:
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "svm",
        SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            probability=True,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )
    )
])

svm_model.fit(X_train, y_train)

svm_pred = svm_model.predict(X_test)
svm_score = svm_model.predict_proba(X_test)[:, 1]

svm_result = evaluate_model(
    "SVM",
    y_test,
    svm_pred,
    svm_score
)


## 5. ANN classifier

An MLP neural network provides the requested ANN comparison while using exactly the same distance/similarity feature representation.

The architecture is intentionally small so that the project remains computationally manageable.


In [ ]:
ann_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "ann",
        MLPClassifier(
            hidden_layer_sizes=(32, 16),
            activation="relu",
            solver="adam",
            alpha=1e-4,
            learning_rate_init=1e-3,
            max_iter=100,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=10,
            random_state=RANDOM_STATE
        )
    )
])

ann_model.fit(X_train, y_train)

ann_pred = ann_model.predict(X_test)
ann_score = ann_model.predict_proba(X_test)[:, 1]

ann_result = evaluate_model(
    "ANN",
    y_test,
    ann_pred,
    ann_score
)


## 6. Threshold-based baseline

As a simple interpretable baseline, classify a window as pre-sepsis when its maximum similarity to any mined discriminative pattern is at least a selected threshold.

The threshold is learned from the training data only.

This avoids choosing the threshold using the test set.


In [ ]:
threshold_feature = "max_pattern_similarity"

if threshold_feature not in X_train.columns:
    raise ValueError(f"Missing {threshold_feature}")

positive_train_scores = X_train.loc[y_train == 1, threshold_feature]

# Use the median positive training score as a simple, reproducible threshold.
baseline_threshold = float(positive_train_scores.median())

baseline_score = X_test[threshold_feature].to_numpy()
baseline_pred = (baseline_score >= baseline_threshold).astype(int)

baseline_result = evaluate_model(
    f"Pattern similarity threshold ({baseline_threshold:.3f})",
    y_test,
    baseline_pred,
    baseline_score
)

print("Threshold learned from training positive windows:", baseline_threshold)


## 7. Compare model results

These are the initial test-set results. Notebook 06 will provide the more complete comparison, visualizations, and ablation experiments.


In [ ]:
results_df = pd.DataFrame([
    svm_result,
    ann_result,
    baseline_result
])

display(
    results_df[
        [
            "model",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "auroc",
            "auprc"
        ]
    ].sort_values("auprc", ascending=False)
)


## 8. Confusion matrices

Confusion matrices provide an interpretable view of false positives and false negatives.


In [ ]:
model_predictions = {
    "SVM": svm_pred,
    "ANN": ann_pred,
    "Threshold baseline": baseline_pred
}

for name, predictions in model_predictions.items():
    cm = confusion_matrix(y_test, predictions)

    plt.figure(figsize=(5, 4))
    plt.imshow(cm)
    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.xticks([0, 1], ["Non-sepsis", "Pre-sepsis"])
    plt.yticks([0, 1], ["Non-sepsis", "Pre-sepsis"])

    for i in range(2):
        for j in range(2):
            plt.text(j, i, cm[i, j], ha="center", va="center")

    plt.colorbar()
    plt.tight_layout()

    safe_name = name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    plt.savefig(
        EVAL_DIR / f"confusion_matrix_{safe_name}.png",
        dpi=200,
        bbox_inches="tight"
    )
    plt.show()


## 9. Save models and results

The trained models and the exact split information are saved so later evaluation can be reproduced.


In [ ]:
import joblib

joblib.dump(svm_model, MODEL_DIR / "svm_model.joblib")
joblib.dump(ann_model, MODEL_DIR / "ann_model.joblib")

results_df.to_csv(
    EVAL_DIR / "classification_results.csv",
    index=False
)

split_indices = {
    "train_indices": X_train.index.tolist(),
    "test_indices": X_test.index.tolist(),
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "feature_columns": similarity_columns
}

with open(EVAL_DIR / "classification_split.json", "w", encoding="utf-8") as f:
    json.dump(split_indices, f, indent=2)

with open(MODEL_DIR / "baseline_threshold.txt", "w", encoding="utf-8") as f:
    f.write(str(baseline_threshold))

print("Saved:")
print(" -", MODEL_DIR / "svm_model.joblib")
print(" -", MODEL_DIR / "ann_model.joblib")
print(" -", EVAL_DIR / "classification_results.csv")
print(" -", EVAL_DIR / "classification_split.json")
print(" -", MODEL_DIR / "baseline_threshold.txt")


## 10. Sanity checks

The notebook is ready for Notebook 06 when:
- all models train successfully;
- AUROC and AUPRC are finite;
- predictions contain both valid binary labels;
- results are saved.


In [ ]:
assert len(results_df) == 3
assert np.isfinite(results_df[["auroc", "auprc"]].to_numpy()).all()
assert set(svm_pred).issubset({0, 1})
assert set(ann_pred).issubset({0, 1})
assert set(baseline_pred).issubset({0, 1})

print("All classification sanity checks passed.")
